# Data Preparation Notebook 2: Create a CSV to train embedding model
- Create a CSV file with `variable_para` and `alternate_variable_para` 
- use semantically interoperable variable names, definitions and values from `definitions.ncit.csv` and `synonyms.ncit.csv` files 

In [ ]:
# imports
import pandas as pd
import numpy as np
from itertools import combinations
from itertools import zip_longest

### Read definitions

In [ ]:
definitions_df = pd.read_csv('definitions.ncit.csv', index_col='concept')

In [ ]:
definitions_df.head(n=3)

In [ ]:
definitions_df.shape

### 181270 unique concepts combed through NCIt APIs 
- definitions obtained for these
- 212234 unique concepts exist in NCIt file

In [ ]:
definitions_df.index.nunique()

In [ ]:
181270/212234

### Read synonyms

In [ ]:
synonyms_df = pd.read_csv('synonyms.ncit.csv', index_col='concept')

In [ ]:
synonyms_df.head(n=3)

In [ ]:
synonyms_df.shape

### How many definitions per concept are there?

In [ ]:
definition_count = definitions_df.groupby('concept')['definition'].agg('count')

In [ ]:
definition_count.index

In [ ]:
definition_count.head(n=3)

In [ ]:
definition_count.value_counts()

In [ ]:
definitions_df.loc[definition_count == 1].shape

In [ ]:
definitions_df.loc[definition_count > 1].shape

In [ ]:
multiple_definition_df = definitions_df.loc[definition_count > 1]

In [ ]:
multiple_definition_df.shape

In [ ]:
multiple_definition_df.head(n=3)

## Group synonyms by concept to add variable names 
- for each concept, get the list of variable names
- add to above df

In [ ]:
syn_by_concept = synonyms_df.groupby(level=0)['name'].agg(list)

In [ ]:
syn_by_concept.head(n=3)

In [ ]:
syn_by_concept.name = 'variable_name'

In [ ]:
syn_by_concept.shape

### Join variable names list with multiple_definition_df

In [ ]:
multiple_definitions_with_varnames = pd.merge(
    left=multiple_definition_df,
    right=syn_by_concept,
    left_index=True,
    right_index=True,
    how='inner'
)

In [ ]:
multiple_definitions_with_varnames.shape

In [ ]:
multiple_definitions_with_varnames.head(n=3)

### Explode variable_name column
- we want to have only one variable_name per row

In [ ]:
multiple_definitions_with_varnames = multiple_definitions_with_varnames.reset_index().explode("variable_name")

In [ ]:
multiple_definitions_with_varnames.head(n=5)

In [ ]:
multiple_definitions_with_varnames.shape

In [ ]:
multiple_definitions_with_varnames = multiple_definitions_with_varnames.set_index('concept')

### Out of 180K unique concepts how many have multiple definitions
- only 51K

In [ ]:
multiple_definitions_with_varnames.index.nunique()

### Create semantically interoperable definitions for the 51K concepts
- group by concept
- for each concept group, iterate through the list and make unique pairs of concept, definition, source and variable_name


In [ ]:
rows = []
# group by concept
for concept, group in multiple_definitions_with_varnames.groupby(level=0):
    # convert group to list
    group = list(group.iterrows())
    # iterate through list and make pairs
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            # first row
            r1 = group[i][1]
            # second row
            r2 = group[j][1]
            rows.append({
                "concept": concept,
                "definition_1": r1["definition"],
                "source_1": r1["source"],
                "variable_name_1": r1["variable_name"],
                "definition_2": r2["definition"],
                "source_2": r2["source"],
                "variable_name_2": r2["variable_name"],
            })
unique_pairs = pd.DataFrame(rows)

In [ ]:
unique_pairs.head()

In [ ]:
unique_pairs.shape

In [ ]:
# drop duplicate rows
unique_pairs_nodup = unique_pairs.drop_duplicates()

In [ ]:
unique_pairs_nodup.head(n=3)

In [ ]:
unique_pairs_nodup.shape

### remove rows that have same info, but different case

In [ ]:
mask = (
    unique_pairs_nodup["variable_name_1"].str.casefold().eq(unique_pairs_nodup["variable_name_2"].str.casefold()) &
    unique_pairs_nodup["definition_1"].str.casefold().eq(unique_pairs_nodup["definition_2"].str.casefold())
)

unique_pairs_nodup = unique_pairs_nodup[~mask]

In [ ]:
unique_pairs_nodup.shape

### check rows where:
- definition_1, variable_name_1 == definition_2, variable_name_2
- filter them out

In [ ]:
unique_pairs_nodup_filtered = unique_pairs_nodup[
    (unique_pairs_nodup['definition_1'] != unique_pairs_nodup['definition_2']) & (unique_pairs_nodup['variable_name_1'] != unique_pairs_nodup['variable_name_2'])
]

In [ ]:
unique_pairs_nodup_filtered.shape

In [ ]:
unique_pairs_nodup_filtered.head(n=3)

### for each concept, further remove redundancies
- only keep rows where variable_name_1, definition_1 and variable_name_2, definition_2 are truly unique
- i.e in example below, we want to remove second row
```
concept,def1,name1,def2,name2
X,a,b,c,d
X,c,d,b,a
```

In [ ]:
def remove_redundancies(df):
    pairs = df.apply(
        lambda r: sorted([
            (str(r["variable_name_1"]), str(r["definition_1"])),
            (str(r["variable_name_2"]), str(r["definition_2"]))
        ]),
        axis=1
    )

    # add a pair1 and pair2 for filtering later
    df[["pair1", "pair2"]] = pd.DataFrame(
        pairs.tolist(), index=df.index
    )

    # drop dups
    df = (
        df.drop_duplicates(subset=["concept", "pair1", "pair2"])
          .drop(columns=["pair1", "pair2"])
    )
    return df

In [ ]:
unique_name_def_df = remove_redundancies(unique_pairs_nodup_filtered)

In [ ]:
unique_name_def_df.shape

### Get values for each concept
 - we will comb "values" from the thesaurus file
 - values are defined as child concepts for a parent concept

In [ ]:
thesaurus_df = pd.read_csv('Thesaurus_26.06e.txt', sep='\t')

In [ ]:
thesaurus_df.head(n=2)

### Values = child concepts for a given code
- to get child concepts, for each code,
- look for rows where code == 'parents'. All concepts in the returned rows are child concepts
- we can then look for the concepts in the synonyms df and capture the variable names as child values

#### example, lets look at Race
- look for rows where parents == C17049
- all rows returned are child concepts or "values" or C17049

In [ ]:
thesaurus_df[thesaurus_df['parents'] == 'C17049']

In [ ]:
thesaurus_df[thesaurus_df['parents'] == 'C17049'].shape

In [ ]:
parent_child_df = thesaurus_df.groupby('parents')['code'].agg(list)

In [ ]:
parent_child_df.head(n=2)

In [ ]:
parent_child_df.loc['C17049']

#### synonyms df has all child values, i.e variable_names for each code
- e.g. C104495 from the list for race above

In [ ]:
synonyms_df.loc['C104495', ]

In [ ]:
def remove_value_redundancies(df):
    pairs = df.apply(
        lambda r: tuple(sorted([r["value_list_1"], r["value_list_2"]])),
        axis=1
    )
    df["pair"] = pairs
    result = (
        df.drop_duplicates(subset=["parent_concept_code", "pair"])
          .drop(columns="pair")
    )
    return result
    

In [ ]:
def build_positional_value_lists(groups):
    if not groups:
        return []
    max_len = max(len(g) for g in groups)
    positional_lists = []
    for pos in range(max_len):
        col = []
        for group in groups:
            col.append(group[pos] if pos < len(group) else group[-1])
        positional_lists.append(col)
    return positional_lists
 

In [ ]:
def to_quoted_csv_list(values):
    return ", ".join(f"'{v}'" for v in values)

def pair_up_lists(positional_lists):
    # we want only value_list_1 and value_list_2 in each row
    # for even number of lists we can split
    # if odd, we will drop
    pairs = []
    for i in range(0, len(positional_lists) - 1, 2):
        pairs.append((positional_lists[i], positional_lists[i + 1]))
    return pairs

def positional_lists_to_dataframe(positional_lists, parent_concept):
    pairs = pair_up_lists(positional_lists)
    rows = [
        {"parent_concept_code": parent_concept, "value_list_1": to_quoted_csv_list(col1), "value_list_2": to_quoted_csv_list(col2)}
        for col1, col2 in pairs
    ]
    return pd.DataFrame(rows)

In [ ]:
def create_values_df(parent_concept):
    # get all child codes
    codes = parent_child_df.loc[parent_concept]
    groups = []
    for code in codes:
        vals = list(set(synonyms_df.loc[code]['name'].values))
        groups.append(vals)
    positional_lists = build_positional_value_lists(groups)
    df = positional_lists_to_dataframe(positional_lists, parent_concept)
    return df

In [ ]:
# test
create_values_df('C17049')

In [ ]:
parent_child_df.head(n=3)

In [ ]:
for parent_concept in list(parent_child_df.index):
    # print('processing:', parent_concept)
    if '|' in parent_concept:
        processed_parent_concept = parent_concept.replace("|", "_")
    else:
        processed_parent_concept = parent_concept
    values_output_file = '/'.join(['values_outputs', 
                                   processed_parent_concept + '.values.csv'
                                  ])
    try:
        create_values_df(parent_concept).to_csv(values_output_file)
    except Exception as e:
        pass

In [ ]:
print('completed values processing')

In [ ]:
unique_name_def_df.head(n=3)

### Merge value lists with `unique_pairs_nodup_filtered`

In [ ]:
def get_values_df(concept_name):
    if '|' in concept_name:
        processed_concept_name = concept_name.replace("|", "_")
    else:
        processed_concept_name = concept_name
    try:
        values_df = pd.read_csv('/'.join([
            'values_outputs', processed_concept_name + '.values.csv'
        ]))
    except Exception as e:
        values_df = pd.DataFrame()
    return values_df  

In [ ]:
values_df = pd.concat([
    get_values_df(c)
    for c in unique_pairs_nodup_filtered['concept']
])

In [ ]:
values_df.shape

In [ ]:
values_df = values_df.set_index('parent_concept_code')

In [ ]:
values_df.head()

In [ ]:
values_df = values_df.drop(columns=["Unnamed: 0"])

In [ ]:
values_df = values_df.drop_duplicates()

In [ ]:
values_df.shape

In [ ]:
values_df.head()

In [ ]:
# test
values_df.loc['C17049',]

In [ ]:
unique_name_def_df[unique_name_def_df['concept'] == 'C17049']

In [ ]:
values_df = values_df.reset_index()

In [ ]:
values_df.head(n=3)

In [ ]:
unique_name_def_df_with_values = pd.merge(
    left=unique_name_def_df,
    right=values_df,
    left_on='concept',
    right_on='parent_concept_code',
    how='left'
)

In [ ]:
unique_name_def_df.shape

In [ ]:
values_df.shape

In [ ]:
# only 4166 variables with semantically interop values
values_df['parent_concept_code'].nunique()

In [ ]:
unique_name_def_df_with_values.shape

In [ ]:
unique_name_def_df_with_values[unique_name_def_df_with_values['concept'] == 'C17049']

In [ ]:
unique_name_def_df_with_values.columns

In [ ]:
unique_name_def_df_with_values = unique_name_def_df_with_values.rename(columns={
    'definition_1': 'variable_description_1',
    'definition_2': 'variable_description_2',
    'value_list_1': 'variable_value_list_1',
    'value_list_2': 'variable_value_list_2'
})

In [ ]:
unique_name_def_df_with_values.head(n=3)

In [ ]:
unique_name_def_df_with_values.columns

In [ ]:
unique_name_def_df_with_values["variable_para"] = unique_name_def_df_with_values.apply(
    lambda x: '.'.join([
        str(x['variable_name_1']) if pd.notna(x['variable_name_1']) else '',
        str(x['variable_description_1']) if pd.notna(x['variable_description_1']) else '',
        str(x['variable_value_list_1']) if pd.notna(x['variable_value_list_1']) else '',
    ]).strip(),
    axis=1
)

In [ ]:
unique_name_def_df_with_values["alternate_variable_para"] = unique_name_def_df_with_values.apply(
    lambda x: '.'.join([
        str(x['variable_name_2']) if pd.notna(x['variable_name_2']) else '',
        str(x['variable_description_2']) if pd.notna(x['variable_description_2']) else '',
        str(x['variable_value_list_2']) if pd.notna(x['variable_value_list_2']) else '',
    ]).strip(),
    axis=1
)

In [ ]:
# check
unique_name_def_df_with_values[unique_name_def_df_with_values['concept'] == 'C17049'].to_csv('test.csv')

In [ ]:
# output to file
unique_name_def_df_with_values.to_csv('ncit_training.csv')

In [ ]:
### NOTE: definitions from NCIt may already be very similar, we havent filtered them out